# Õppetund 11 - Mudeli konteksti protokoll (MCP)

**Mudeli konteksti protokoll (MCP)** on avatud standard, mis võimaldab agentidel dünaamiliselt avastada ja kasutada tööriistu, ressursse ja andmeallikaid käituse ajal. Selle asemel, et tööriistu agendi sisse kõvakodeerida, laseb MCP agentidel ühendada end väliste serveritega, mis pakuvad võimeid nõudmisel.

Selles õppetükis õpid:
- Mis on MCP ja miks see on agentide süsteemide jaoks oluline
- Kuidas töötab MCP klient-server arhitektuur
- Kuidas ehitada agente, kes kasutavad MCP-stiilis tööriistade avastamist


## Seadistamine

**Eeldused:**
- Microsoft Foundry projekt koos juurutatud mudeliga
- Autentimiseks käivita `az login`


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mis on mudeli konteksti protokoll (MCP)?

MCP määratleb standardse viisi, kuidas tehisintellekti agendid avastavad ja suhtlevad väliste tööriistade ning andmeallikatega:

- **MCP server**: Pakub tööriistu, ressursse ja juhiseid standardse protokolli kaudu
- **MCP klient**: Agendi käitusaeg, mis ühendub serveritega ja avastab saadavalolevaid võimalusi
- **Dünaamiline avastamine**: Agendid ei vaja kõvasti kodeeritud tööriistu — nad avastavad, mis käitusajal saadaval on

See on võimas võimalus laiendatavate agendisüsteemide ehitamiseks, kus uusi võimekusi saab lisada ilma agendi koodi muutmata.


## Kuidas MCP töötab

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (MCP klient) ühendub MCP serveriga
2. Server vastab kättesaadavate tööriistade ja nende skeemide nimekirjaga
3. Agent saab seejärel kutsuda suvalist leitud tööriista oma mõtlemise käigus
4. Tulemused voolavad tagasi sama protokolli kaudu


## MCP tööriista avastamise simuleerimine

Kuna tõeline MCP server nõuab käivitatud serveriprotsessi, demonstreerime mustrit, kasutades `@tool` funktsioone, mis simuleerivad, mida MCP-ga ühendatud majutusteenus pakuks.

Tootmises leitakse need tööriistad dünaamiliselt MCP serverist, mitte ei määratleta kohapeal.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Agent MCP-stiilis tööriistadega loomine


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP tootmiskeskkonnas

Tootmiskeskkonnas võimaldab MCP võimsaid mustreid:

- **Dünaamiline tööriistade avastamine**: Agendid ühenduvad MCP serveritega ja avastavad tööriistad jooksvalt
- **Eraldatud arhitektuur**: Tööriistapakkujad saavad iseseisvalt agendist iseseisvalt uuendada
- **Organisatsioonidevaheline jagamine**: Meeskonnad saavad MCP serverite kaudu pakkuda võimalusi, mida iga agent saab kasutada
- **Microsoft Agendi raamistikutugi**: MAF-il on sisseehitatud MCP klienditugi `mcp` integratsiooni kaudu

Tõelise MCP serveri kasutamiseks MAF-iga ühenduksite läbi `hosted_mcp_tool()` või MCP kliendi integratsiooni.

**Lisateave:**
- [MCP spetsifikatsioon](https://modelcontextprotocol.io/)
- [Microsoft Agendi raamistik MCP tugi](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

In this lesson, you learned:
- **MCP** is an open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** lets agents discover capabilities at runtime
- MCP enables **extensible, decoupled agent systems** where tools can be added without code changes
- Microsoft Agent Framework provides **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
